# SO-101 multi-workspace control panel (LeRobot v3)

Menu: **Workspace 1** (screw, ACT) · **Workspace 2** (hammer / wrench, SmolVLA; text · voice · vision) · **Workspace 3** (burger / Thor's hammer, SmolVLA; text · voice).

Run the cells **top to bottom**. Nothing moves until you press a button. Keep a hand on the **hardware e-stop**; the red **STOP** button always works.

Before the first real run: `workspaces.yaml` filled in, `mock: true` tested once (steps 1-4 work without any hardware), robot calibrated with the same `robot.id` used for recording.

## 1. Environment (run once; skip if already installed)
Needs **Python ≥ 3.12**. Same LeRobot commit as the training box.

In [11]:
# Uncomment on a fresh machine (in a terminal is better):
# !pip install "lerobot[feetech,remote,core_scripts] @ git+https://github.com/huggingface/lerobot.git@146d8474932c8b566717eb2738ba459337ab71c4"
# !pip install ipywidgets pyserial pyyaml            # panel + rail
# !pip install sounddevice faster-whisper            # only for voice input
# !pip install opencv-python mediapipe               # only for vision input
import sys; print(sys.version)

  Cloning https://github.com/huggingface/lerobot.git (to revision 146d8474932c8b566717eb2738ba459337ab71c4) to /tmp/pip-install-quw1odb7/lerobot_618d069b2cba420caefef3611f6015bf
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/lerobot.git /tmp/pip-install-quw1odb7/lerobot_618d069b2cba420caefef3611f6015bf
  Running command git rev-parse -q --verify '146d8474932c8b566717eb2738ba459337ab71c4^{commit}'
  146d8474932c8b566717eb2738ba459337ab71c4
  Resolved https://github.com/huggingface/lerobot.git to commit 146d8474932c8b566717eb2738ba459337ab71c4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of lerobot to determine which version is compatible with other requirements. This could take a while.
ERROR: Package 'lerobot' requires a different Python: 3.10.12 not in '>=3.12'
3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]


## 2. Load config and run the pre-flight check
Choose which input methods you intend to use; only their packages are checked/imported.

In [15]:
import importlib, panel_core; importlib.reload(panel_core)
from panel_core import *

cfg = load_config("workspaces.yaml")
METHODS = ("text", "voice", "vision")          # remove the ones you do not want to use, e.g. ("text",)

problems = preflight(cfg, METHODS)
print("MOCK MODE (no hardware)" if cfg["mock"] else "REAL HARDWARE")
print("\n".join("  - " + p for p in problems) if problems else "pre-flight OK")

MOCK MODE (no hardware)
  - Python 3.10: LeRobot 0.6 needs >= 3.12


## 3. Build the controller and the panel

In [16]:
import ipywidgets as w
from IPython.display import display

log_lines = []
log_box = w.Textarea(layout=w.Layout(width="100%", height="260px"), disabled=True)
def logf(msg):
    log_lines.append(f"{time.strftime('%H:%M:%S')}  {msg}")
    log_box.value = "\n".join(log_lines[-300:])

state_lbl = w.HTML()
SEL_ANYTIME = w.VBox()          # workspace-specific controls are rebuilt on every state change

def on_change(state, ws):
    colour = {"READY": "green", "RUNNING": "#c60", "FAULT": "red"}.get(state, "#555")
    name = next((x["name"] for x in cfg["workspaces"] if x["id"] == ws), "-")
    state_lbl.value = f"<b>State:</b> <span style='color:{colour}'>{state}</span> &nbsp; <b>Workspace:</b> {ws} ({name})"
    rebuild(state, ws)

ctl = Controller(cfg, logf=logf, on_change=on_change)

btn_ws = [w.Button(description=f"Workspace {x['id']}: {x['name']}", button_style="primary",
                   layout=w.Layout(width="260px", height="60px")) for x in cfg["workspaces"]]
for b, x in zip(btn_ws, cfg["workspaces"]):
    b.on_click(lambda _, i=x["id"]: ctl.select_workspace(i))

btn_stop = w.Button(description="STOP", button_style="danger", layout=w.Layout(width="120px", height="60px"))
btn_stop.on_click(lambda _: (logf("STOP pressed"), ctl.stop()))
btn_back = w.Button(description="Back / change workspace", layout=w.Layout(width="200px", height="60px"))
btn_back.on_click(lambda _: ctl.back())
btn_ack = w.Button(description="Acknowledge fault", button_style="warning", layout=w.Layout(width="200px"))
btn_ack.on_click(lambda _: ctl.acknowledge_fault())

voice, gesture = None, None
preview = w.Image(format="jpeg", width=320)

def stop_inputs():
    global voice, gesture
    if voice: voice.stop(); voice = None
    if gesture: gesture.stop(); gesture = None

def make_input_controls(wsc):
    """Input-method chooser for the selected workspace (text / voice / vision as allowed by the config)."""
    allowed = [m for m in wsc["inputs"] if m in METHODS]
    if not wsc["inputs"]:                       # workspace 1: just one model
        b = w.Button(description="Start screwing", button_style="success", layout=w.Layout(width="220px", height="50px"))
        key = next(iter(wsc["options"]))
        b.on_click(lambda _: ctl.choose(key, "button"))
        return [b]
    method = w.ToggleButtons(options=allowed, value=None, description="Input:")
    area = w.VBox()
    opts = list(wsc["options"])

    def pick(change):
        stop_inputs()
        m = change["new"]
        if m == "text":
            bs = []
            for k in opts:
                b = w.Button(description=k, button_style="success", layout=w.Layout(width="200px", height="50px"))
                b.on_click(lambda _, k=k: ctl.choose(k, "text")); bs.append(b)
            tb = w.Text(placeholder=f"type e.g. '{wsc['options'][opts[0]]['keywords'][0]}' and press Enter", layout=w.Layout(width="420px"))
            def submit(t):
                k = parse_keywords(t.value, wsc["options"]); logf(f"text: {t.value!r} -> {k}")
                if k: ctl.choose(k, "text")
                t.value = ""
            tb.on_submit(submit)
            area.children = [w.HBox(bs), tb]
        elif m == "voice":
            global voice
            logf("loading speech model ...")
            voice = VoiceInput(ctl, logf, cfg["voice"]["whisper_model"])
            voice.start(cfg["voice"]["prompt_by_workspace"].get(wsc["id"], ""))
            area.children = [w.HTML("🎤 Listening. Say: " + " or ".join(f"<b>{k}</b>" for k in opts))]
        elif m == "vision":
            global gesture
            gesture = GestureInput(ctl, logf, cfg["vision"]["camera_index"], "hand_landmarker.task",
                                   on_frame=lambda b: setattr(preview, "value", b))
            gesture.start()
            m_txt = ", ".join(f"{o['gesture']} finger(s) = {k}" for k, o in wsc["options"].items())
            area.children = [w.HTML(f"✋ Hold still for 0.6 s: {m_txt}. Relax your hand to re-arm."), preview]
    method.observe(pick, names="value")
    return [method, area]

def rebuild(state, ws):
    if state == "IDLE":
        stop_inputs()
        SEL_ANYTIME.children = [w.HTML("<h3>Select a workspace</h3>"), w.HBox(btn_ws)]
    elif state == "READY":
        wsc = next(x for x in cfg["workspaces"] if x["id"] == ws)
        SEL_ANYTIME.children = [w.HTML(f"<h3>Workspace {ws}: {wsc['name']}</h3>"), *make_input_controls(wsc), w.HBox([btn_back, btn_stop])]
    elif state == "FAULT":
        stop_inputs()
        SEL_ANYTIME.children = [w.HTML("<h3 style='color:red'>FAULT - read the log, make the arm/rail safe, then acknowledge</h3>"), btn_ack]
    else:   # busy: only STOP is available
        SEL_ANYTIME.children = [w.HTML(f"<h3>{state.title()} ...</h3>"), btn_stop]

on_change("IDLE", None)

## 4. The panel
Press a workspace button. Order of events: **arm → rest pose → rail moves → input menu → model loads → runs → arm returns to rest → ready**.

In [17]:
display(state_lbl, SEL_ANYTIME, w.HTML("<b>Log</b>"), log_box)

HTML(value="<b>State:</b> <span style='color:#555'>IDLE</span> &nbsp; <b>Workspace:</b> None (-)")

HTML(value='<b>Log</b>')

Textarea(value='', disabled=True, layout=Layout(height='260px', width='100%'))

## 5. Shut down (always run this when finished)

In [10]:
stop_inputs(); ctl.close(); logf("closed")

### Troubleshooting
| Symptom | Fix |
|---|---|
| `FAULT: rollout exited during load` | read the `|` lines in the log: wrong robot port/id, camera index, or the model repo / task string |
| Rail moves to wrong place | check `rail.positions` units; test `SerialRail` alone first (see README step D) |
| Remote: `policy server did not become ready` | the Brev instance must be **Running**; run `bash ~/serve/run_server.sh hammer` there by hand and read the error |
| Voice hears nothing | `python -m sounddevice` lists input devices; set the default input in the OS |
| Arm drops when idle | `robot.disable_torque_on_disconnect=false` is already passed; verify the rest pose is gravity-stable |